# Analysis 6 (draft): does the Dutch team's own tournament run explain the shape?

**Status: draft, not yet critiqued.** This notebook is Stage 4's "build" step for
Analysis 6 in `analysis-log.md`, following on from feedback item 1 on Analysis 5's
chart (`notebooks/analysis/feedback-football-tournaments.md`). It has **not** been
through Stage 5 (critique) yet — that needs a real person looking at the rendered
chart and giving an honest first-impression read, the same way every earlier
analysis in this project did it. Run this top to bottom, look at the result, then
go back to `analysis-log.md` / the goad conversation to do Stage 5 for real.

**What this builds, per the stage 1-4 answers already recorded in the log:**
- Extends Analysis 5's three tournament panels (Euro 2020, World Cup 2022,
  Euro 2024) with a fourth: **World Cup 2026** (dates web-verified, see below).
- Adds a confirmed table of all 19 Netherlands matches across the four
  tournaments, joined onto the daily message-count table as `is_nl_match_day`.
- Marks every NL match day as a dashed vertical line on its tournament's panel,
  so it's visually inspectable whether that day shows a spike over baseline.
- **Deliberately left out for this pass** (per Stage 2/3): `match_result`
  (win/draw/loss) — the question here is only whether match days themselves see
  more activity, not whether winning/losing changes the size of the effect.
  Parked as a named follow-up (the student's own suspicion: losses might spike
  *more* than wins, "easier to bash the team than to celebrate").


In [ ]:
import pandas as pd
from goad_toolkit.datatransforms import FlagDates, Pipeline, RollingAvg
from goad_toolkit.visualizer import FacetPlot, LinePlot, PlotSettings

from wa_analyzer.data import load_own_chat

own = load_own_chat()
# same fix as 03-football-tournaments.ipynb: this processed file stores
# `timestamp` as plain strings, not datetime -- resample() needs a real
# DatetimeIndex.
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)


## Tournament reference table, now with World Cup 2026

Same static-lookup pattern as `analysis-log.md`'s other reference tables --
approximate, validate further before relying on it beyond a first draft.
Official start/end dates for the first three tournaments are unchanged from
`03-football-tournaments.ipynb`. **World Cup 2026 dates are new, web-verified**
(2026-06-11 to 2026-07-19, opening match in Mexico City, final at MetLife
Stadium) -- see `analysis-log.md`, Analysis 6, Stage 4 for the source link.


In [ ]:
tournaments = [
    {"name": "Euro 2020", "start": "2021-06-11", "end": "2021-07-11"},
    {"name": "World Cup 2022", "start": "2022-11-20", "end": "2022-12-18"},
    {"name": "Euro 2024", "start": "2024-06-14", "end": "2024-07-14"},
    {"name": "World Cup 2026", "start": "2026-06-11", "end": "2026-07-19"},
]


## Netherlands match table -- student-confirmed, web-sourced

All 19 NL matches across the four tournaments (Stage 2). Web-sourced (Wikipedia,
UEFA, FIFA, ESPN, Fox Sports, Al Jazeera), then checked against the student's own
memory, which corrected one date (World Cup 2026 vs. Morocco: moved from the
initially-found 2026-06-29 to the confirmed 2026-06-30). `match_result` is
included here only as a comment for future reference -- it is **not** turned
into a feature or used in the plot below, per Stage 2/3's explicit decision to
leave it out of this pass.


In [ ]:
nl_matches = [
    # Euro 2020 (played 2021), Group C
    {"date": "2021-06-13", "tournament": "Euro 2020", "opponent": "Ukraine"},        # W 3-2
    {"date": "2021-06-17", "tournament": "Euro 2020", "opponent": "Austria"},        # W 2-0
    {"date": "2021-06-21", "tournament": "Euro 2020", "opponent": "North Macedonia"},# W 3-0
    {"date": "2021-06-27", "tournament": "Euro 2020", "opponent": "Czech Republic"}, # L 0-2, eliminated
    # World Cup 2022, Group A
    {"date": "2022-11-21", "tournament": "World Cup 2022", "opponent": "Senegal"},   # W 2-0
    {"date": "2022-11-25", "tournament": "World Cup 2022", "opponent": "Ecuador"},   # D 1-1
    {"date": "2022-11-29", "tournament": "World Cup 2022", "opponent": "Qatar"},     # W 2-0
    {"date": "2022-12-03", "tournament": "World Cup 2022", "opponent": "United States"}, # W 3-1
    {"date": "2022-12-09", "tournament": "World Cup 2022", "opponent": "Argentina"}, # D 2-2 aet, lost pens, eliminated
    # Euro 2024, Group D
    {"date": "2024-06-16", "tournament": "Euro 2024", "opponent": "Poland"},         # W 2-1
    {"date": "2024-06-21", "tournament": "Euro 2024", "opponent": "France"},         # D 0-0
    {"date": "2024-06-25", "tournament": "Euro 2024", "opponent": "Austria"},        # L 2-3, still advanced
    {"date": "2024-07-02", "tournament": "Euro 2024", "opponent": "Romania"},        # W 3-0
    {"date": "2024-07-06", "tournament": "Euro 2024", "opponent": "Turkey"},         # W 2-1
    {"date": "2024-07-10", "tournament": "Euro 2024", "opponent": "England"},        # L 1-2, eliminated
    # World Cup 2026, Group F
    {"date": "2026-06-14", "tournament": "World Cup 2026", "opponent": "Japan"},     # D 2-2
    {"date": "2026-06-20", "tournament": "World Cup 2026", "opponent": "Sweden"},    # W 5-1
    {"date": "2026-06-25", "tournament": "World Cup 2026", "opponent": "Tunisia"},   # W 3-1
    {"date": "2026-06-30", "tournament": "World Cup 2026", "opponent": "Morocco"},   # D 1-1 aet, lost pens, eliminated
]

nl_matches_df = pd.DataFrame(nl_matches)
nl_matches_df["date"] = pd.to_datetime(nl_matches_df["date"])
assert len(nl_matches_df) == 19, "expected 19 NL matches across the four tournaments"
nl_matches_df


## Join `is_nl_match_day` onto the daily message-count table

Exact-date-only join -- student's explicit call, not extending a few days after
each match for post-match reaction/aftermath (a candidate for a future pass).


In [ ]:
daily = own.set_index("timestamp").resample("D").size().rename("messages").reset_index()
global_baseline = daily["messages"].mean()

daily["is_nl_match_day"] = daily["timestamp"].isin(nl_matches_df["date"])
print(f"{daily['is_nl_match_day'].sum()} of {len(nl_matches_df)} NL match days found in the chat's date range")


## Build the four per-tournament panels

Same approach as `03-football-tournaments.ipynb`: rolling average computed on the
*full* daily series first (so the 7-day window has real history at every
tournament boundary), then sliced per tournament with context padding either
side. `is_nl_match_day` rides along in the slice.


In [ ]:
smoothed = Pipeline().add(
    RollingAvg, column="messages", window=7, rename=True
).apply(daily)

context_pad = pd.Timedelta(days=21)
panels = []
for t in tournaments:
    start = pd.Timestamp(t["start"])
    end = pd.Timestamp(t["end"])
    during_range = pd.date_range(start, end)

    flagged = (
        Pipeline()
        .add(FlagDates, column="timestamp", dates=during_range, feature="in_during")
        .apply(smoothed.copy())
    )

    panel = flagged[
        (flagged["timestamp"] >= start - context_pad)
        & (flagged["timestamp"] <= end + context_pad)
    ].copy()
    panel["tournament"] = t["name"]
    panels.append(panel)

combined = pd.concat(panels, ignore_index=True)

print(f"overall daily baseline: {global_baseline:.1f} messages/day")
combined.groupby("tournament")["is_nl_match_day"].sum()


## Plot: four small-multiple panels, NL match days marked

Same family and layout Analysis 5 already used (time series, one panel per
tournament, shared y-scale) -- extended to four panels, with a dashed vertical
line on every one of that tournament's NL match days (`VerticalDate`, drawn in
red by default). Labels stated once, on the first panel only, matching Analysis
5's "state it once" convention -- same colour coding holds on every panel.

**This is the raw build. Stage 5 (critique) has not happened yet** -- look at
this chart yourself before reading anything into it.


In [ ]:
order = [t["name"] for t in tournaments]

football = PlotSettings(
    figsize=(18, 4.5),
    title="Do Netherlands match days spike above the tournament's own rise?",
    xlabel="",
    ylabel="messages per day",
    subplot_ylabels=["messages per day", "", "", ""],
    sharey=True,
    max_cols=4,
)

facet = FacetPlot(football)
fig, axes = facet.plot(
    inner=LinePlot(football), data=combined, by="tournament", order=order,
    x="timestamp", y="messages", color="#cccccc",
)
fig.get_layout_engine().set(rect=(0, 0.08, 1, 0.93))

label_box = dict(facecolor="white", alpha=0.75, edgecolor="none", pad=1.5)

for i, (ax, t) in enumerate(zip(axes, tournaments)):
    panel = combined[combined["tournament"] == t["name"]]
    facet.plot_on_axes(
        LinePlot(football), ax, data=panel, x="timestamp",
        y="messages_rolling_avg", color="crimson",
    )
    start, end = pd.Timestamp(t["start"]), pd.Timestamp(t["end"])
    # Stage 5 critique fix: the shaded "during tournament" background band is
    # gone -- it read as the same red as the match-day lines, grouping two
    # conceptually different things (the whole window vs. one match) together.
    ax.axhline(global_baseline, color="black", linestyle="--", linewidth=1)

    # Stage 5 critique round 2: `VerticalDate` always draws linewidth=2 at
    # full opacity, on top of everything else -- that's exactly what made the
    # match lines dominate the data lines. Plain `axvline` here instead: thin,
    # semi-transparent, and zorder=1 so it sits BEHIND the grey/crimson lines
    # (their default zorder is 2) rather than in front of them.
    match_dates = nl_matches_df.loc[nl_matches_df["tournament"] == t["name"], "date"]
    for match_date in match_dates:
        ax.axvline(match_date, color="red", linestyle="--", linewidth=1,
                   alpha=0.35, zorder=1)

    ax.set_xticks([])
    ax.set_xlabel("")

    if i == 0:
        ax.text(start - pd.Timedelta(days=18), global_baseline + 2,
                "chat's overall\ndaily average", ha="left", va="bottom",
                fontsize=8, color="black", bbox=label_box)
        first_match = match_dates.min()
        ax.text(first_match, ax.get_ylim()[1] * 0.92, "NL match day",
                ha="left", va="top", fontsize=8.5, color="red", alpha=0.7,
                bbox=label_box)

fig.text(
    0.5, 0.015,
    "Dashed red lines mark every Netherlands match in that tournament "
    "(drawn thin, semi-transparent, behind the data lines). Stage 5 critique "
    "round 2 applied (analysis-log.md, Analysis 6).",
    ha="center", va="bottom", fontsize=8.5, color="#444444", style="italic",
)

## Focus chart: Euro 2024 only

Requested after critique round 2 -- Euro 2024 was the only tournament that
looked convincing by eye in Stage 5. A single-panel chart, real x-axis dates
kept (no clutter problem with only one panel), a legend instead of direct
labels (there's only one panel, so a lookup box isn't the cost it was across
four), and the same thin/semi-transparent match-day lines from round 2.


In [ ]:
euro2024 = combined[combined["tournament"] == "Euro 2024"]
euro2024_matches = nl_matches_df.loc[nl_matches_df["tournament"] == "Euro 2024", "date"]

focus = PlotSettings(
    figsize=(12, 4.5),
    title="Euro 2024: Netherlands match days vs. chat activity",
    xlabel="",
    ylabel="messages per day",
    xtick_rotation=45,
)

lines = LinePlot(focus)
fig, ax = lines.plot(data=euro2024, x="timestamp", y="messages",
                     color="#cccccc", label="daily")
lines.plot_on(LinePlot(focus), data=euro2024, x="timestamp",
              y="messages_rolling_avg", color="crimson", label="7-day average")
ax.axhline(global_baseline, color="black", linestyle="--", linewidth=1,
           label="chat's overall daily average")

for i, match_date in enumerate(euro2024_matches):
    ax.axvline(match_date, color="red", linestyle="--", linewidth=1,
               alpha=0.35, zorder=1,
               label="NL match day" if i == 0 else None)

ax.legend()


## Not yet done

- **Stage 5 critique, round 2 applied:** match-day lines switched from
  `VerticalDate` (always linewidth=2, full opacity, drawn on top) to plain
  `axvline` -- thin, `alpha=0.35`, `zorder=1` so they sit behind the
  grey/crimson data lines instead of in front of them. Not yet re-critiqued
  by the student.
- **Euro-2024-only focus chart added**, per the student's request, using the
  same round-2 line styling -- also not yet critiqued.
- **Stage 6 (verification) still hasn't happened.** The student's own Stage
  5 claim was narrower than Stage 1's original proposition -- only Euro
  2024 looked convincing by eye. That needs an actual check (a null, a
  shuffle test, a confounder check), not just a second look at a chart.
- `match_result` (win/draw/loss) is still parked, per Stage 2/3 -- not in
  either chart.
- The exact-date-only join means a match late in the day (evening kickoff)
  and its likely spillover into the next calendar day aren't distinguished
  -- named in Stage 2, not resolved here.
